In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [ ]:
def get_statistics(scores_a, scores_b, margin):
    differences = scores_a - scores_b

    stat, p_value_noninferior = stats.wilcoxon(scores_b + margin, scores_a, alternative='greater')

    mean_diff = np.mean(differences)
    std_diff = np.std(differences, ddof=1)
    cohens_d = mean_diff / std_diff

    correlation, p_value_correlation = stats.pearsonr(scores_a, scores_b, alternative='greater')

    return correlation, p_value_correlation, p_value_noninferior, cohens_d

In [ ]:
metric = 'VUS-PR' # Use 'VUS-PR' for reproducing results in Table 8, 'Event-based-F1' for reproducing results in Table 9.

# The non-inferiority margins for each metric, as mentioned in Appendix J of the paper.
margins = {'VUS-PR': 0.008, 'Event-based-F1': 0.02}

#variants = ['Chronos-Bolt-small-64', 'Chronos-Bolt-small-256', 'Chronos-Bolt-small-512', 'Chronos-Bolt-base-64', 'Chronos-Bolt-base-256', 'Chronos-Bolt-base-512', 'Time-MoE-base-64', 'Time-MoE-base-256', 'Time-MoE-base-512', 'Time-MoE-large-64', 'Time-MoE-large-256', 'Time-MoE-large-512', 'Timesfm-1.0-64', 'Timesfm-1.0-256', 'Timesfm-1.0-512', 'Timesfm-2.0-64', 'Timesfm-2.0-256', 'Timesfm-2.0-512']
variants = ['bolt_small_64', 'bolt_small_256', 'bolt_small_512', 'bolt_base_64', 'bolt_base_256', 'bolt_base_512', 'TimeMoE-50M_64', 'TimeMoE-50M_256', 'TimeMoE-50M_512', 'TimeMoE-200M_64', 'TimeMoE-200M_256', 'TimeMoE-200M_512', 'timesfm-1.0-200m_64', 'timesfm-1.0-200m_256', 'timesfm-1.0-200m_512', 'timesfm-2.0-500m_64', 'timesfm-2.0-500m_256', 'timesfm-2.0-500m_512']
path_baseline = '../results/baselines/last_3.csv'

# Initialize an empty DataFrame with the specified column names and data types

column_specs = {
    'Variant': 'object',
    'mean (TSFM)': 'float64',  
    'mean (baseline)': 'float64',      
    'correlation': 'float64',  
    'p-value (correlation)': 'float64',  
    'p-value (non-inferiority)': 'float64',
    "Cohen's d": 'float64'        
}

df = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in column_specs.items()})

# Iterate over the combinations of model variants and window sizes, read the corresponding results, and compute the statistics.

for index, variant in enumerate(variants):
    path_model = f'../results/tsfms/{variant}.csv'

    results_model = pd.read_csv(path_model)
    results_baseline = pd.read_csv(path_baseline)

    x = results_model[metric].to_numpy()
    y = results_baseline[metric].to_numpy()

    scores_a = x.astype(np.float32)
    scores_b = y.astype(np.float32)
    
    correlation, p_value_correlation, p_value_noninferior, cohens_d = get_statistics(scores_a, scores_b, margins[metric])

    new_row_data = pd.Series({'Variant': '\\textbf{' + variant + '}', 'mean (TSFM)':scores_a.mean(), 'mean (baseline)':scores_b.mean(), 'correlation': correlation, 'p-value (correlation)': p_value_correlation, 'p-value (non-inferiority)' : p_value_noninferior, "Cohen's d": cohens_d})
    new_row_df = new_row_data.to_frame().T
    df = pd.concat([df, new_row_df], ignore_index=True)

df

In [ ]:
# Format the DataFrame for LaTeX output.

format_dict = {col: '{:.3f}' for col in df.select_dtypes(include='object').columns}
format_dict['p-value (correlation)'] = '{:.2e}'
del format_dict['Variant']

print(df[['Variant', 'mean (TSFM)', 'mean (baseline)', 'correlation', 'p-value (correlation)', "Cohen's d"]].style.format(format_dict).hide().to_latex())